# Maintenir des croyances avec un JTMS en C#

**Navigation** · [Sommaire](README.md) · [Version Python](Argumentation-06-JTMS-Python.ipynb) · [ATMS et conflits](Argumentation-06b-ATMS-Conflits-CSharp.ipynb)

Un raisonneur monotone conserve une conclusion lorsqu’on ajoute des prémisses. Un système de croyances doit aussi retirer une conclusion quand son fondement est rétracté. Vous allez représenter les justifications, calculer les étiquettes IN/OUT, tracer les cascades et reconnaître un cycle de soutien sans fondation.

**Provenance** · Le moteur pédagogique conserve les instruments de trace et d’explication distillés du JTMS du tronc EPITA, projet 1.4.1-JTMS de Thomas Leguere. La première partie s’appuie sur la bibliothèque standard C#. Le dernier exemple verse les croyances dans le véritable état partagé du cursus avec Python.NET.

**Prérequis** · Graphe orienté, implication et raisonnement propositionnel. Run local sous .NET Interactive et CPython 3.13.

In [1]:
#r "nuget: pythonnet,3.1.0"
#load "../../Config/PythonNetLoader.cs"
#nullable enable
using System;
using System.Collections.Generic;
using System.Linq;
using System.Text.Json;
using Python.Runtime;

static void RequireArgument(bool ok, string message)
{
    if (!ok)
        throw new InvalidOperationException(message);
}

// Le package local expose ses moteurs sans outil distant. Ajouter le dossier du carnet au chemin du CPython embarqué.
if (PythonNetLoader.Initialize(typeof(PythonEngine).Assembly, "json")is not null)
    PythonEngine.BeginAllowThreads();
using (Py.GIL())
{
    using var boot = Py.CreateScope();
    boot.Exec("import sys, os; sys.path.insert(0, os.getcwd())");
    using var package = Py.Import("argumentation_lib");
    using var version = package.GetAttr("__version__");
    Console.WriteLine($"argumentation_lib {version.As<string>()} via Python.NET 3.1.0, .NET {Environment.Version}.");
}

Installing Packages pythonnet

argumentation_lib 0.2.0 via Python.NET 3.1.0, .NET 10.0.14.


## 1. Déterminer si une justification est active

Une justification relie une conclusion à deux ensembles. Les soutiens $I$ doivent être IN, les défenseurs $O$ doivent être OUT. Une règle est active quand toutes ces conditions tiennent. Une prémisse possède une justification vide. Plusieurs justifications pour une même conclusion sont des soutiens alternatifs.

Les cas porteurs de ce carnet utilisent le support positif. Le calcul itératif part de OUT et ajoute des faits soutenus. Cette croissance est monotone dans le fragment positif, donc bornée par le nombre de croyances. La représentation accepte aussi des conditions OUT. Sur un graphe négatif général, leur état peut changer pendant une itération. Le simple algorithme de ce carnet ne constitue pas une sémantique générale pour ces cycles.

In [2]:
#nullable enable
record Justification(string Conclusion, string[] Support, string[] OutConditions);
record Retraction(string Belief, string? Cause, int Depth);
class Jtms
{
    readonly SortedSet<string> nodes = new(StringComparer.Ordinal);
    readonly List<Justification> rules = [];
    public void Premise(string name) => Add(name, []);
    public void Add(string name, string[] support, string[]? blockers = null)
    {
        blockers ??= [];
        nodes.Add(name);
        nodes.UnionWith(support);
        nodes.UnionWith(blockers);
        rules.Add(new Justification(name, support, blockers));
    }

    static bool Holds(Justification rule, Dictionary<string, bool> status)
    {
        return rule.Support.All(name => status[name]) && rule.OutConditions.All(name => !status[name]);
    }

    public Dictionary<string, bool> Label()
    {
        // Dans le fragment positif, une conclusion promue conserve son témoin de soutien.
        var status = nodes.ToDictionary(name => name, _ => false, StringComparer.Ordinal);
        bool changed;
        do
        {
            changed = false;
            foreach (var rule in rules)
            {
                if (status[rule.Conclusion] || !Holds(rule, status))
                    continue;
                status[rule.Conclusion] = true;
                changed = true;
            }
        }
        while (changed);
        return status;
    }

    public void Retract(string name) => rules.RemoveAll(r => r.Conclusion == name);
    public Retraction[] Chain(string name)
    {
        var before = Label();
        Retract(name);
        var after = Label();
        var queue = new Queue<Retraction>();
        var seen = new HashSet<string>(StringComparer.Ordinal)
        {
            name
        };
        var chain = new List<Retraction>();
        queue.Enqueue(new Retraction(name, null, 0));
        while (queue.TryDequeue(out var step))
        {
            chain.Add(step);
            // Seul un soutien positif définit la cause d'une chute pour cette trace.
            var outgoing = rules.Where(r => r.Support.Contains(step.Belief)).Select(r => r.Conclusion).Distinct().Order(StringComparer.Ordinal);
            foreach (string target in outgoing)
            {
                if (before[target] && !after[target] && seen.Add(target))
                    queue.Enqueue(new Retraction(target, step.Belief, step.Depth + 1));
            }
        }

        return chain.ToArray();
    }

    public string[] Circular()
    {
        var status = Label();
        var dependencies = nodes.ToDictionary(name => name, name => rules.Where(r => r.Conclusion == name).SelectMany(r => r.Support).Distinct().ToArray());
        var indexes = new Dictionary<string, int>();
        var low = new Dictionary<string, int>();
        var stack = new Stack<string>();
        var onStack = new HashSet<string>();
        var circular = new SortedSet<string>(StringComparer.Ordinal);
        int nextIndex = 0;
        void Visit(string name)
        {
            indexes[name] = low[name] = nextIndex++;
            stack.Push(name);
            onStack.Add(name);
            foreach (string other in dependencies[name])
            {
                if (!indexes.ContainsKey(other))
                {
                    Visit(other);
                    low[name] = Math.Min(low[name], low[other]);
                }
                else if (onStack.Contains(other))
                {
                    low[name] = Math.Min(low[name], indexes[other]);
                }
            }

            // Le sommet racine de Tarjan ferme exactement une composante.
            if (low[name] != indexes[name])
                return;
            var component = new List<string>();
            string member;
            do
            {
                member = stack.Pop();
                onStack.Remove(member);
                component.Add(member);
            }
            while (member != name);
            bool cycle = component.Count > 1 || dependencies[name].Contains(name);
            if (cycle && component.All(x => !status[x]))
                circular.UnionWith(component);
        }

        foreach (string name in nodes)
            if (!indexes.ContainsKey(name))
                Visit(name);
        return circular.ToArray();
    }

    public string[][] Supports(string name) => rules.Where(r => r.Conclusion == name).Select(r => r.Support).ToArray();
    public string Explain(string name)
    {
        var status = Label();
        var loops = Circular();
        string State(string x) => status[x] ? "IN" : "OUT";
        var related = rules.Where(r => r.Conclusion == name).ToArray();
        var lines = new List<string>
        {
            $"{name} = {State(name)}"};
        foreach (var rule in related)
        {
            string verdict = Holds(rule, status) ? "valide" : loops.Contains(name) ? "non concluante (circularité)" : "invalide";
            string Info(string[] entries) => entries.Length == 0 ? "-" : string.Join(", ", entries.Select(x => $"{x} ({State(x)})"));
            lines.Add($"  IN=[{Info(rule.Support)}] OUT=[{Info(rule.OutConditions)}] : {verdict}");
        }

        if (related.Length == 0)
            lines.Add("  aucune justification");
        // Remonter seulement par des soutiens OUT et s'arrêter quand un cycle a déjà été visité.
        var broken = new SortedSet<string>(StringComparer.Ordinal);
        var seen = new HashSet<string>();
        void Collect(string x)
        {
            if (status[x] || !seen.Add(x))
                return;
            var rs = rules.Where(r => r.Conclusion == x).ToArray();
            if (rs.Length == 0)
                broken.Add(x);
            foreach (string dependency in rs.SelectMany(r => r.Support))
                Collect(dependency);
        }

        Collect(name);
        if (broken.Count > 0)
            lines.Add("  fondation perdue : " + string.Join(", ", broken));
        return string.Join(Environment.NewLine, lines);
    }

    public void Print()
    {
        foreach (var(name, inState)in Label())
            Console.WriteLine($"{name, -18} {(inState ? "IN" : "OUT")}");
        var loops = Circular();
        if (loops.Length > 0)
            Console.WriteLine("Circularité sans fondation : " + string.Join(", ", loops));
    }
}

Console.WriteLine("Moteur JTMS C# chargé : étiquetage, trace en largeur, Tarjan et explication.");

Moteur JTMS C# chargé : étiquetage, trace en largeur, Tarjan et explication.


## 2. Une chaîne ancrée

A est une prémisse. B exige A et C exige B. Les deux règles deviennent successivement actives et toutes les croyances finissent IN.

In [3]:
static Jtms ChainAbc()
{
    var t = new Jtms();
    // Les deux justifications transmettent le soutien de la prémisse le long de la chaîne.
    t.Premise("A");
    t.Add("B", ["A"]);
    t.Add("C", ["B"]);
    return t;
}

var chain = ChainAbc();
RequireArgument(chain.Label().Values.All(x => x), "Une chaîne ancrée doit être IN.");
Console.WriteLine("A → B → C");
chain.Print();

A → B → C


A                  IN


B                  IN


C                  IN


### Exercice 1 — Ajouter un soutien alternatif

Ajoutez D comme prémisse et faites de D un second soutien de C. Rétractez A puis prédisez les étiquettes de B et C. Une justification réunissant B et D demanderait une autre condition.

In [4]:
#nullable enable
static Jtms? SoutienAlternatif(Jtms t)
{
    // TODO étudiant : ajouter un autre soutien autonome à la même conclusion.
    return null;
}

var alt = SoutienAlternatif(ChainAbc());
if (alt is not null)
{
    alt.Retract("A");
    RequireArgument(alt.Label()["C"] && !alt.Label()["B"], "C doit conserver D après la chute de B.");
    alt.Print();
}
else
    Console.WriteLine("À compléter : ajouter D puis vérifier le soutien alternatif.");

À compléter : ajouter D puis vérifier le soutien alternatif.


## 3. Rétracter et suivre une cascade

Retirer une prémisse retire ses justifications. On recalcule l’étiquetage global. La trace compare le graphe avant et après et parcourt les soutiens tombés en largeur depuis le déclencheur. Elle donne la profondeur et une cause immédiate pour chaque retrait.

In [5]:
var shrinking = ChainAbc();
Console.WriteLine("Avant rétractation");
shrinking.Print();
Retraction[] steps = shrinking.Chain("A");
// La trace est un témoin ordonné du changement de statut, pas la liste de toutes les arêtes.
RequireArgument(steps.Select(s => s.Belief).SequenceEqual(new[] { "A", "B", "C" }) && steps[0].Cause is null && steps[1] == new Retraction("B", "A", 1) && steps[2] == new Retraction("C", "B", 2), "La cascade attendue est A, puis B, puis C.");
foreach (var step in steps)
    Console.WriteLine($"{step.Depth} | {step.Belief} | {step.Cause ?? "déclencheur"}");
Console.WriteLine("Après rétractation");
shrinking.Print();

Avant rétractation


A                  IN


B                  IN


C                  IN


0 | A | déclencheur


1 | B | A


2 | C | B


Après rétractation


A                  OUT


B                  OUT


C                  OUT


### Lecture de la cascade

A était la seule fondation. Chaque règle aval a perdu un soutien nécessaire, ce qui explique la nouvelle étiquette de B puis de C. La profondeur de la trace distingue un retrait direct de ses conséquences transitives.

### Exercice 2 — Une cascade en diamant

Construisez A→B, A→C, puis une justification conjointe B et C→E. Prédisez le niveau de E avant de rétracter A. Comparez les étiquettes après retrait, et expliquez pourquoi une des deux causes immédiates suffit dans la trace.

In [6]:
#nullable enable
static Jtms? ConstruireDiamant()
{
    // TODO étudiant : déclarer le graphe et une seule règle conjonctive vers E.
    return null;
}

var diamond = ConstruireDiamant();
if (diamond is not null)
{
    var trace = diamond.Chain("A");
    RequireArgument(trace.Any(x => x.Belief == "E" && x.Depth == 2) && !diamond.Label()["E"], "E doit tomber au niveau 2.");
    diamond.Print();
}
else
    Console.WriteLine("À compléter : comparer la trace du diamant à votre prédiction.");

À compléter : comparer la trace du diamant à votre prédiction.


## 4. Détecter un cycle de soutien non fondé

Si A exige B et B exige A, aucune règle ne peut partir sans une prémisse extérieure. Tarjan repère les composantes fortement connexes du graphe de soutien. On signale ici une composante cyclique quand tous ses membres restent OUT, y compris les auto-boucles. Une composante avec soutien externe n’est pas signalée selon ce critère.

In [7]:
var circular = new Jtms();
circular.Add("A", ["B"]);
circular.Add("B", ["A"]);
// Le cycle seul ne fournit à aucun de ses membres une justification activée.
RequireArgument(circular.Circular().SequenceEqual(new[] { "A", "B" }) && circular.Label().Values.All(x => !x), "Les deux croyances circulaires doivent rester OUT.");
circular.Print();

A                  OUT


B                  OUT


Circularité sans fondation : A, B


### Exercice 3 — Cycle à trois croyances

Construisez A qui exige B, B qui exige C, C qui exige A. Vérifiez les trois étiquettes et la composante. Ajoutez ensuite une fondation externe et examinez ce qu’elle change.

In [8]:
#nullable enable
static Jtms? BoucleSansFondation()
{
    // TODO étudiant : trois justifications à soutien unique, sans prémisse.
    return null;
}

var loop = BoucleSansFondation();
if (loop is not null)
{
    RequireArgument(loop.Circular().SequenceEqual(new[] { "A", "B", "C" }), "Composante circulaire attendue.");
    loop.Print();
}
else
    Console.WriteLine("À compléter : construire et inspecter le cycle à trois.");

À compléter : construire et inspecter le cycle à trois.


## 5. Appliquer la trace à un appel à l’autorité

Un témoignage suffit provisoirement à accepter X, qui soutient une décision. Si la prémisse est disqualifiée comme appel à l’autorité, le JTMS n’évalue pas l’expertise lui-même. Il reçoit la rétractation et explique pourquoi ses deux conclusions perdent leur soutien.

In [9]:
static Jtms ExpertClaim()
{
    var t = new Jtms();
    // Le statut du témoignage vient de l'analyse argumentative, non du calcul de points fixes.
    t.Premise("expert_claims_X");
    t.Add("X_is_true", ["expert_claims_X"]);
    t.Add("decide_X", ["X_is_true"]);
    return t;
}

var claim = ExpertClaim();
Console.WriteLine("Avant retrait du témoignage");
claim.Print();
claim.Retract("expert_claims_X");
string explanation = claim.Explain("decide_X");
RequireArgument(explanation.Contains("fondation perdue : expert_claims_X") && claim.Label().Values.All(x => !x), "L'explication doit remonter à la fondation effectivement retirée.");
Console.WriteLine(explanation);
claim.Print();

Avant retrait du témoignage


X_is_true          IN


decide_X           IN


expert_claims_X    IN


decide_X = OUT
  IN=[X_is_true (OUT)] OUT=[-] : invalide
  fondation perdue : expert_claims_X


X_is_true          OUT


decide_X           OUT


expert_claims_X    OUT


## 6. Écrire dans l’état de l’analyse partagée

Les autres étapes de l’argumentation utilisent le même conteneur pour leurs détections et verdicts. On prend les trois croyances avant retrait et on verse dans le véritable UnifiedAnalysisState leur nom, validité et ensembles de supports. Le pont transfère ce résultat structuré sans modifier le mécanisme du JTMS.

In [10]:
var clean = ExpertClaim();
var beliefs = clean.Label().Select(item => new { Name = item.Key, Valid = item.Value, Supports = clean.Supports(item.Key) }).ToArray();
using (Py.GIL())
{
    using var scope = Py.CreateScope();
    using var payload = new PyString(JsonSerializer.Serialize(beliefs));
    scope.Set("payload", payload);
    // L'état est celui qui est effectivement réutilisé par les autres séances du cursus.
    scope.Exec("""
        import json
        from argumentation_lib import UnifiedAnalysisState
        state = UnifiedAnalysisState(initial_text="Argument modèle pour le pont JTMS")
        for belief in json.loads(payload):
            state.add_jtms_belief(name=belief["Name"], valid=belief["Valid"], justifications=belief["Supports"])
        encoded = json.dumps(state.jtms_beliefs, ensure_ascii=False, sort_keys=True)
        """);
    string encoded = scope.Get<string>("encoded");
    using var parsed = JsonDocument.Parse(encoded);
    var stored = parsed.RootElement.EnumerateObject().Select(x => x.Value).ToArray();
    RequireArgument(stored.Length == 3 && stored.All(x => x.GetProperty("valid").GetBoolean()) && stored.Select(x => x.GetProperty("name").GetString()).Order().SequenceEqual(beliefs.Select(x => x.Name).Order()) && stored.All(x => JsonSerializer.Deserialize<string[][]>(x.GetProperty("justifications").GetRawText())!.Select(a => string.Join("|", a)).SequenceEqual(clean.Supports(x.GetProperty("name").GetString()!).Select(a => string.Join("|", a)))), "L'état partagé ne retrouve pas les trois croyances.");
    foreach (var item in stored)
        Console.WriteLine(item.GetRawText());
}

{"justifications": [["expert_claims_X"]], "name": "X_is_true", "valid": true}


{"justifications": [["X_is_true"]], "name": "decide_X", "valid": true}


{"justifications": [[]], "name": "expert_claims_X", "valid": true}


## À retenir

- Une croyance IN a au moins une justification active. Une rétractation recalcule les dépendances et la trace identifie les chutes.
- Un cycle sans fondation ne peut produire de soutien depuis l’état initial OUT.
- La trace explique l’effet d’une rétractation, tandis que l’état partagé expose les croyances aux autres étapes du raisonnement.

## Vérifiez votre compréhension

**Pourquoi E tombe-t-il si B et C sont des soutiens conjoints ?**

<details><summary>Réponse</summary>

La justification de E exige que les deux soutiens soient IN. La chute de l’un suffit à désactiver cette règle. E pourrait rester IN si une autre règle devenait active.

</details>

## Pour aller plus loin

- [ATMS et interprétations concurrentes](Argumentation-06b-ATMS-Conflits-CSharp.ipynb).
- J. Doyle, *A Truth Maintenance System*, Artificial Intelligence 12, 1979.